## Part 1: Vector Database Fundamentals

In this notebook we will:
1. Connect to a local Qdrant instance
2. Load an e-commerce product dataset from HuggingFace
3. Embed products with OpenAI and load them into Qdrant
4. Run semantic searches, filtered searches, and RAG

**Key concepts**: embeddings, vector dimensions, cosine similarity, payload filtering, RAG.

## Setup and Dependencies

In [ ]:
# If you ran pip install for requirements, do not uncomment this cell
#!pip install qdrant-client openai python-dotenv datasets

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    print(f"Qdrant URL: {QDRANT_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

## Connect to Qdrant

We are connecting to a local Qdrant instance running via Docker Compose on port 6333.

In [ ]:
from qdrant_client import QdrantClient

# Connecting to our local instance of our vector database
qdrant = QdrantClient(url=QDRANT_URL)

# Let's have a look at what collections we have in our database
info = qdrant.get_collections()
print(f"Connected to Qdrant. Existing collections: {[c.name for c in info.collections]}")

## Embedding Helper

Unlike some vector databases, Qdrant does not have a built-in vectorizer when you add objects to the database. We will use OpenAI's embedding API ourselves, vectorize our objects and then pass them to the database.

In [ ]:
import openai

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Creating a functionb to vectorize our objects in batch
def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts using OpenAI."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

# Creating a function to vectorize a single object for testing
def embed_text(text: str) -> list[float]:
    """Embed a single text string using OpenAI."""
    return embed_texts([text])[0]

# Quick test
test_vec = embed_text("hello world")
print(f"Embedding dimension: {len(test_vec)}")

## Load the Dataset

We will use Qdrant's [H&M e-commerce products dataset](https://huggingface.co/datasets/Qdrant/hm_ecommerce_products) from HuggingFace. It has 105K products with rich metadata including product names, descriptions, colors, departments, categories, and more. 

We will take a 2,500 product sample and embed it ourselves into a new collection in memory.

In [ ]:
from datasets import load_dataset

# Pull in the data set from Hugging Face
full_dataset = load_dataset("Qdrant/hm_ecommerce_products", split="train")

# Showing data set info
print(f"Full dataset size: {len(full_dataset)}")
print(f"Columns: {full_dataset.column_names}")

In [ ]:
# Let's look at specific dta for one product
sample = full_dataset[0]
print(f"Product: {sample['prod_name']}")
print(f"Type: {sample['product_type_name']}")
print(f"Group: {sample['product_group_name']}")
print(f"Color: {sample['colour_group_name']}")
print(f"Department: {sample['department_name']}")
print(f"Section: {sample['section_name']}")
print(f"Index: {sample['index_name']}")
print(f"Description: {sample['detail_desc']}")
print(f"\nText used for embedding:\n{sample['text_to_embed']}")

In [ ]:
# Take 2,500 products
dataset = full_dataset.select(range(2500))
print(f"Working with {len(dataset)} products")

## Embed the Products

We embed the `text_to_embed` field, a pre-built string that combines product name, type, color, department, and description into a single passage. We batch the API calls for speed.

In [ ]:
# A nice tool for us to watch our embedding progress
from tqdm import tqdm

# Setting up our batches to embed
BATCH_SIZE = 100
all_texts = [item["text_to_embed"] for item in dataset]
all_vectors = []

# Pushing our objects to OpenAI for embedding in batches
for i in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding with OpenAI"):
    batch = all_texts[i:i + BATCH_SIZE]
    vectors = embed_texts(batch)
    all_vectors.extend(vectors)

print(f"Embedded {len(all_vectors)} products, dimension: {len(all_vectors[0])}")

## Create the Collection and Import

In Qdrant, a **collection** is the equivalent of a database table. Each item is a **point** with an ID, a vector, and a **payload** (object properties you can filter on).

In [ ]:
from qdrant_client.models import Distance, VectorParams, PointStruct

# Naming our Collection
COLLECTION_NAME = "products"

# A BAD idea to run this in production
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

# Creating our collection with correct metadata
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=1536,  # OpenAI text-embedding-3-small dimension
        distance=Distance.COSINE, # Any idea why cosine??? If so let me know!
    ),
)

print(f"Collection '{COLLECTION_NAME}' created (1536-dim, cosine)")

In [ ]:
# Creating a list of points, or records, to add to our HNSW index (vector store)
points = []
for i, item in enumerate(dataset):
    points.append(PointStruct(
        # This is very important, any idea why the ID matters?
        id=i,
        vector=all_vectors[i],
        payload={
            "prod_name": item["prod_name"],
            "product_type_name": item["product_type_name"],
            "product_group_name": item["product_group_name"],
            "colour_group_name": item["colour_group_name"],
            "department_name": item["department_name"],
            "section_name": item["section_name"],
            "index_name": item["index_name"],
            "detail_desc": item["detail_desc"],
            "garment_group_name": item["garment_group_name"],
        },
    ))

# Upload in batches of 500
BATCH_SIZE = 500
for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"  Uploaded {min(i + BATCH_SIZE, len(points))}/{len(points)} points")

# Sanity check
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"\nImported {count} products")

## Semantic Search

Vector search finds results based on meaning. We embed the query with the same OpenAI model and find the closest products by cosine similarity.

In [ ]:
# Embedding our query with the same model we used to embed our data
query_vector = embed_text("dark winter jacket")

# Sending our embedded query to the database
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
).points

# Display our top 5 results
for i, hit in enumerate(results, 1):
    print(f"{i}. {hit.payload['prod_name']}")
    print(f"   Type: {hit.payload['product_type_name']} | Color: {hit.payload['colour_group_name']}")
    print(f"   Score: {hit.score:.4f}")
    print(f"   {hit.payload['detail_desc'][:100]}...")
    print()

## Filtered Search

Payload filters constrain the search space before vector similarity runs. This is critical for production, you almost always want to narrow your results and hav eteh ability to utilize your property data to find the nuance not caght by semantic search.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Vectorize our query
query_vector = embed_text("casual summer top")

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    # Add our property filter
    query_filter=Filter(
        must=[
            FieldCondition(key="colour_group_name", match=MatchValue(value="White")),
        ]
    ),
    limit=5,
).points

for i, hit in enumerate(results, 1):
    print(f"{i}. {hit.payload['prod_name']} ({hit.payload['colour_group_name']})")
    print(f"   Type: {hit.payload['product_type_name']} | Score: {hit.score:.4f}")
    print()

### Combining Multiple Filters

Combine filters with `must` (AND), `should` (OR), and `must_not` (NOT).

In [ ]:
query_vector = embed_text("comfortable basics")

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key="index_name", match=MatchValue(value="Ladieswear")),
        ],
        must_not=[
            FieldCondition(key="colour_group_name", match=MatchValue(value="Black")),
        ]
    ),
    limit=5,
).points

for i, hit in enumerate(results, 1):
    print(f"{i}. {hit.payload['prod_name']}")
    print(f"   {hit.payload['section_name']} | {hit.payload['colour_group_name']} | Score: {hit.score:.4f}")
    print()

## RAG: Retrieve and Generate

Qdrant handles retrieval. For generation, we pass the retrieved context to an LLM. This is basic RAG at its core.

In [ ]:
# Create our query
query = "I need a formal outfit for a business meeting"

# Vectorize it
query_vector = embed_text(query)

# Pull similar objects from ouyr database
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
).points

# Build context from retrieved products
context = "\n".join(
    f"- {hit.payload['prod_name']} ({hit.payload['product_type_name']}, "
    f"{hit.payload['colour_group_name']}): {hit.payload['detail_desc']}"
    for hit in results
)

# Send our original query with our query results and a system prompt to our generative model
response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a personal shopping assistant. Recommend products based on the available inventory."},
        {"role": "user", "content": f"Customer request: {query}\n\nAvailable products:\n{context}\n\nRecommend the best options and explain why."},
    ],
)

print("=== Shopping Assistant ===")
print(response.choices[0].message.content)

## Summary

What we learned:

- **Embedding is the translation step** — turning text into vectors that capture meaning
- **Vector search** finds semantically similar results by comparing these vectors
- **Payload filters** constrain the search space with metadata (`must`, `should`, `must_not`)
- **RAG** = retrieve context from the vector database, then generate answers with an LLM
- **Your query embedding model must match your document embedding model** — you can't mix and match

Next up: we will build agents that automate the hard parts,  dynamic filtering and query decomposition.

In [ ]:
qdrant.close()